In [13]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
PROCESSED_DIR = Path(r"E:\Credit Risk Assessment System\dataset\processed")

In [14]:
USE_FEATURE_SUBSET = True  # False = use all cleaned features

try:
    df = pd.read_csv(PROCESSED_DIR / "cleaned_credit_data.csv")
except FileNotFoundError as exc:
    raise FileNotFoundError("Run 03_Data_Cleaning_and_Feature_Engineering.ipynb first") from exc

print("Shape:", df.shape)
display(df.dtypes.value_counts())
display(df["TARGET"].value_counts(normalize=True).mul(100).round(2))

Shape: (307511, 119)


float64    63
int64      41
object     15
Name: count, dtype: int64

TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64

In [15]:
subset_path = PROCESSED_DIR / "selected_top50.csv"

if USE_FEATURE_SUBSET and subset_path.exists():
    selected_features = pd.read_csv(subset_path)["feature"].tolist()
    print(f"Using recommended subset: {len(selected_features)} features")
else:
    selected_features = [c for c in df.columns if c != "TARGET"]
    print(f"Using all cleaned features: {len(selected_features)} features")

X = df[selected_features]
y = df["TARGET"]

print("Feature count:", X.shape[1])
print(selected_features)

Using recommended subset: 50 features
Feature count: 50
['EXT_SOURCE_1', 'EXT_SOURCE_3', 'EXT_SOURCE_2', 'GOODS_CREDIT_RATIO', 'DAYS_BIRTH', 'NAME_EDUCATION_TYPE', 'DAYS_EMPLOYED', 'AGE', 'YEARS_EMPLOYED', 'REGION_RATING_CLIENT_W_CITY', 'DAYS_ID_PUBLISH', 'NAME_INCOME_TYPE', 'REGION_RATING_CLIENT', 'NAME_HOUSING_TYPE', 'EMPLOYMENT_AGE_RATIO', 'ANNUITY_CREDIT_RATIO', 'DAYS_LAST_PHONE_CHANGE', 'CODE_GENDER', 'CREDIT_PER_CHILD', 'AMT_GOODS_PRICE', 'OCCUPATION_TYPE', 'FLAG_DOCUMENT_3', 'DAYS_REGISTRATION', 'REGION_POPULATION_RELATIVE', 'FLOORSMAX_AVG', 'AMT_CREDIT', 'FLOORSMAX_MEDI', 'APARTMENTS_MEDI', 'APARTMENTS_AVG', 'FLOORSMAX_MODE', 'ORGANIZATION_TYPE', 'APARTMENTS_MODE', 'FLAG_EMP_PHONE', 'TOTALAREA_MODE', 'LIVINGAREA_MODE', 'AMT_ANNUITY', 'LIVINGAREA_MEDI', 'LIVINGAREA_AVG', 'BASEMENTAREA_AVG', 'BASEMENTAREA_MEDI', 'BASEMENTAREA_MODE', 'ANNUITY_INCOME_RATIO', 'HOUR_APPR_PROCESS_START', 'REG_CITY_NOT_WORK_CITY', 'FLAG_OWN_CAR', 'LOG_AMT_CREDIT', 'NAME_TYPE_SUITE', 'CNT_FAM_MEMBERS', 

In [16]:
def identify_feature_types(data: pd.DataFrame) -> tuple:
    """Split columns into numerical and categorical lists."""
    categorical = data.select_dtypes(include="object").columns.tolist()
    numerical = data.select_dtypes(include=np.number).columns.tolist()
    return numerical, categorical

numerical_features, categorical_features = identify_feature_types(X)
print(f"Numerical: {len(numerical_features)} | Categorical: {len(categorical_features)}")
print("Categorical columns:", categorical_features)

Numerical: 42 | Categorical: 8
Categorical columns: ['NAME_EDUCATION_TYPE', 'NAME_INCOME_TYPE', 'NAME_HOUSING_TYPE', 'CODE_GENDER', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE', 'FLAG_OWN_CAR', 'NAME_TYPE_SUITE']


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
display(pd.DataFrame({
    "train_%": y_train.value_counts(normalize=True).mul(100).round(2),
    "test_%": y_test.value_counts(normalize=True).mul(100).round(2),
}))

Train: (246008, 50) Test: (61503, 50)


,train_%,test_%
TARGET,,
0,91.93,91.93
1,8.07,8.07


In [18]:
encoder = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numerical_features),
    ]
)

encoder.fit(X_train)  # fit ONLY on training data — prevents leakage
encoded_columns = encoder.get_feature_names_out()

X_train_enc = pd.DataFrame(encoder.transform(X_train), columns=encoded_columns, index=X_train.index)
X_test_enc = pd.DataFrame(encoder.transform(X_test), columns=encoded_columns, index=X_test.index)

print("Encoded shape — train:", X_train_enc.shape, "test:", X_test_enc.shape)

Encoded shape — train: (246008, 149) test: (61503, 149)


In [19]:
print("Before SMOTE:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_enc, y_train)

print("After SMOTE:")
display(y_train_res.value_counts(normalize=True).mul(100).round(2))
print("Train shape after SMOTE:", X_train_res.shape)

Before SMOTE:


TARGET
0    91.93
1     8.07
Name: proportion, dtype: float64

After SMOTE:


TARGET
0    50.0
1    50.0
Name: proportion, dtype: float64

Train shape after SMOTE: (452296, 149)


In [20]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_res), columns=X_train_res.columns, index=X_train_res.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_enc), columns=X_test_enc.columns, index=X_test_enc.index
)

# Unscaled version: same encoded data, scaling skipped
X_train_unscaled = X_train_res.copy()
X_test_unscaled = X_test_enc.copy()

print("Scaled train/test:", X_train_scaled.shape, X_test_scaled.shape)
print("Unscaled train/test:", X_train_unscaled.shape, X_test_unscaled.shape)

Scaled train/test: (452296, 149) (61503, 149)
Unscaled train/test: (452296, 149) (61503, 149)


In [21]:
checks = {
    "no missing values (train scaled)": X_train_scaled.isnull().sum().sum() == 0,
    "no missing values (test scaled)": X_test_scaled.isnull().sum().sum() == 0,
    "matching feature dims (train vs test)": X_train_scaled.shape[1] == X_test_scaled.shape[1],
    "scaler fit only on train (no test rows in scaler)": scaler.n_samples_seen_ == len(X_train_res),
    "class balance after SMOTE (~50/50)": abs(y_train_res.mean() - 0.5) < 0.01,
    "train/test sizes ~80/20": abs(len(X_test) / (len(X_train) + len(X_test)) - 0.2) < 0.01,
}

for check, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {check}")

[PASS] no missing values (train scaled)
[PASS] no missing values (test scaled)
[PASS] matching feature dims (train vs test)
[PASS] scaler fit only on train (no test rows in scaler)
[PASS] class balance after SMOTE (~50/50)
[PASS] train/test sizes ~80/20


In [22]:
output_dir = Path(r"E:\Credit Risk Assessment System\dataset") / "processed_models"
output_dir.mkdir(parents=True, exist_ok=True)

try:
    X_train_scaled.to_csv(output_dir / "X_train_scaled.csv", index=False)
    X_test_scaled.to_csv(output_dir / "X_test_scaled.csv", index=False)
    X_train_unscaled.to_csv(output_dir / "X_train_unscaled.csv", index=False)
    X_test_unscaled.to_csv(output_dir / "X_test_unscaled.csv", index=False)
    y_train_res.to_csv(output_dir / "y_train.csv", index=False)
    y_test.to_csv(output_dir / "y_test.csv", index=False)
    print("Saved 6 files to", output_dir)
except OSError as exc:
    raise OSError(f"Could not save processed data to {output_dir}") from exc

Saved 6 files to E:\Credit Risk Assessment System\dataset\processed_models


In [23]:
metadata = pd.DataFrame([{
    "original_shape": str(df.shape),
    "selected_feature_count": X.shape[1],
    "train_samples": X_train_res.shape[0],
    "test_samples": X_test_enc.shape[0],
    "categorical_features": len(categorical_features),
    "numerical_features": len(numerical_features),
    "encoding_method": "OneHotEncoder",
    "scaling_method": "StandardScaler (Version A only)",
    "smote_applied": True,
    "random_state": RANDOM_STATE,
}])

metadata.to_csv(output_dir / "preprocessing_summary.csv", index=False)
display(metadata)

,original_shape,selected_feature_count,train_samples,test_samples,categorical_features,numerical_features,encoding_method,scaling_method,smote_applied,random_state
0,"(307511, 119)",50,452296,61503,8,42,OneHotEncoder,StandardScaler (Version A only),True,42


# MD:
### 13. Final Summary

| Step | Purpose | Output | Future Usage |
|---|---|---|---|
| Train/test split (80/20, stratified) | Honest evaluation on unseen data | `X_train`/`X_test`, `y_train`/`y_test` | All models |
| One-hot encoding (fit on train only) | Convert categoricals to numeric, no leakage | `X_train_enc`, `X_test_enc` | All models |
| SMOTE (train only) | Balance ~8% minority class | `X_train_res`, `y_train_res` | All models' training data |
| StandardScaler (fit on train only) | Normalize feature magnitude | `X_train_scaled`, `X_test_scaled` | **Logistic Regression** |
| Unscaled encoded data | Preserve natural feature scale | `X_train_unscaled`, `X_test_unscaled` | **Decision Tree, Random Forest, XGBoost** |

- **Logistic Regression** → use `X_train_scaled.csv` / `X_test_scaled.csv`
- **Decision Tree / Random Forest / XGBoost** → use `X_train_unscaled.csv` / `X_test_unscaled.csv`
- All models share the same `y_train.csv` / `y_test.csv`